# 02 · Cleaning and analysis decisions

**Question:** Which records can support reliable comparisons, and what must remain under review?

This notebook follows inspection with explicit, reversible decisions. Original files are immutable.
The primary demand source is the [MTA hourly ridership publication](https://data.ny.gov/Transportation/MTA-Bus-Hourly-Ridership-Beginning-2025/gxb3-akrn).
Daily files are publisher-side sums of that source, not a second passenger survey.
Counts are published boarding estimates, not unique people or vehicle occupancy. Transfers are already included.

Run cells in order in a fresh kernel. Each transformation includes visible checks,
saved results and an explanation of its effect on the analysis.

In [1]:
from pathlib import Path
import sys, json, hashlib, zipfile
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/collect_new_york.py').exists())
sys.path.insert(0, str(ROOT / 'scripts'))
RAW = ROOT / 'data/raw/new_york'
OUT = ROOT / 'data/processed/new_york'
REPORT = ROOT / 'reports/eda'
OUT.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)
print('Project located; analysis uses local, frozen source files.')

Project located; analysis uses local, frozen source files.


## 1. Freeze provenance before changing representations

Verify every original file against its receipt. Store a new manifest for the derived products.
Missing or altered source files stop the run rather than silently refreshing a mutable URL.

In [2]:
sources = []
for receipt_path in sorted(RAW.rglob('*.receipt.json')):
    receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
    path = ROOT / receipt['local_path']
    with path.open('rb') as handle:
        digest = hashlib.file_digest(handle, 'sha256').hexdigest()
    assert digest == receipt['sha256'] and path.stat().st_size == receipt['bytes']
    sources.append({'file': receipt['local_path'], 'sha256': digest, 'bytes': receipt['bytes'],
                    'source_url': receipt['url'], 'retrieved_utc': receipt['retrieved_utc']})
source_register = pd.DataFrame(sources)
source_register.to_csv(REPORT / 'source_register.csv', index=False)
print('Verified source files:', len(source_register))
display(source_register.groupby(source_register.file.str.split('/').str[3]).agg(
    files=('file', 'size'), bytes=('bytes', 'sum')))

Verified source files: 57


,files,bytes
file,,
gtfs,11,97865077
history_daily,14,15204633
ridership,29,47183107
sources,3,336579


## 2. Read the complete demand extracts

Preserve route codes as strings. Rename the daily query's `service_date` alias to `payment_date`:
it describes the date of the publisher's local payment hour, not a GTFS service day.

In [3]:
daily_raw = pd.concat([pd.read_csv(p, dtype={'bus_route': 'string'})
                      for p in sorted((RAW / 'history_daily').glob('*_route_day.csv'))], ignore_index=True)
hourly_raw = pd.concat([pd.read_csv(p, dtype={'bus_route': 'string'})
                       for p in sorted((RAW / 'ridership').glob('*_route_hour.csv'))], ignore_index=True)
print('Daily source:', daily_raw.shape, '| Hourly source:', hourly_raw.shape)
display(daily_raw.head(5))
display(hourly_raw.head(5))

Daily source: (275896, 6) | Hourly source: (993600, 5)


,service_date,bus_route,published_boardings,published_transfers,reported_hours,source_rows
0,2025-01-01T00:00:00.000,B1,5246,864,24,576
1,2025-01-01T00:00:00.000,B100,711,185,24,576
2,2025-01-01T00:00:00.000,B101,0,0,24,576
3,2025-01-01T00:00:00.000,B103,2423,438,24,576
4,2025-01-01T00:00:00.000,B106,0,0,24,576


,transit_timestamp,bus_route,ridership,transfers,source_rows
0,2026-07-01T00:00:00.000,B1,26,6,26
1,2026-07-01T00:00:00.000,B100,8,5,26
2,2026-07-01T00:00:00.000,B101,0,0,26
3,2026-07-01T00:00:00.000,B102,0,0,26
4,2026-07-01T00:00:00.000,B103,31,15,26


## 3. Distinguish exact duplicates from conflicting observations

Exact copies may be deduplicated. Every version of a conflicting route/date or route/hour key
belongs in quarantine; summing conflicts would invent extra boardings. Structural failures
also stop this analysis for investigation rather than being replaced with zero.

In [4]:
from analysis_utils import separate_duplicates
daily, daily_conflicts, daily_duplicate_count = separate_duplicates(daily_raw, ['bus_route', 'service_date'])
hourly, hourly_conflicts, hourly_duplicate_count = separate_duplicates(hourly_raw, ['bus_route', 'transit_timestamp'])
daily_conflicts.to_csv(OUT / 'daily_conflicts.csv', index=False)
hourly_conflicts.to_csv(OUT / 'hourly_conflicts.csv', index=False)
duplicate_audit = pd.DataFrame([
    ['daily', len(daily_raw), daily_duplicate_count, len(daily_conflicts), len(daily)],
    ['hourly', len(hourly_raw), hourly_duplicate_count, len(hourly_conflicts), len(hourly)]
], columns=['table', 'source_rows', 'exact_duplicates_removed', 'conflicting_versions', 'retained_rows'])
display(duplicate_audit)
assert daily_conflicts.empty and hourly_conflicts.empty, 'Review saved conflicts before proceeding'

,table,source_rows,exact_duplicates_removed,conflicting_versions,retained_rows
0,daily,275896,0,0,275896
1,hourly,993600,0,0,993600


In [5]:
daily = daily.rename(columns={'service_date': 'payment_date'})
daily['payment_date'] = pd.to_datetime(daily.payment_date, errors='raise').dt.normalize()
hourly['local_hour'] = pd.to_datetime(hourly.pop('transit_timestamp'), errors='raise')
hourly['payment_date'] = hourly.local_hour.dt.normalize()
assert hourly.local_hour.eq(hourly.local_hour.dt.floor('h')).all()
numeric_columns = {'daily': ['published_boardings', 'published_transfers', 'reported_hours', 'source_rows'],
                   'hourly': ['ridership', 'transfers', 'source_rows']}
domain_rows = []
for name, frame in [('daily', daily), ('hourly', hourly)]:
    assert frame.notna().all().all()
    assert frame.bus_route.str.strip().eq(frame.bus_route).all()
    assert frame.bus_route.str.len().gt(0).all()
    for column in numeric_columns[name]:
        frame[column] = pd.to_numeric(frame[column], errors='raise')
        bad = frame[column].lt(0) | frame[column].mod(1).ne(0)
        domain_rows.append([name, column, int(bad.sum()), int(frame[column].eq(0).sum())])
        assert not bad.any()
assert daily.published_transfers.le(daily.published_boardings).all()
assert hourly.transfers.le(hourly.ridership).all()
display(pd.DataFrame(domain_rows, columns=['table', 'column', 'invalid_values', 'explicit_zeros']))
print('No missing counts were converted to zero; all original count values remain intact.')

,table,column,invalid_values,explicit_zeros
0,daily,published_boardings,0,71125
1,daily,published_transfers,0,72873
2,daily,reported_hours,0,0
3,daily,source_rows,0,0
4,hourly,ridership,0,398896
5,hourly,transfers,0,437185
6,hourly,source_rows,0,0


No missing counts were converted to zero; all original count values remain intact.


## 4. Reconcile totals before applying any quality mask

Use separately downloaded denominator queries from the same publisher; this checks extraction,
not independent passenger measurement. Daily and hourly extracts must agree on their overlap.

In [6]:
denominators = pd.concat([pd.read_csv(p) for p in sorted((RAW / 'history_daily').glob('*_denominator.csv'))])
denominators['payment_date'] = pd.to_datetime(denominators.service_date).dt.normalize()
actual = daily.groupby('payment_date').published_boardings.sum().sort_index()
expected = denominators.set_index('payment_date').published_boardings.sort_index()
pd.testing.assert_series_equal(actual, expected, check_names=False, check_dtype=False)
hour_totals = hourly.groupby('payment_date').ridership.sum().sort_index()
pd.testing.assert_series_equal(actual.reindex(hour_totals.index), hour_totals, check_names=False, check_dtype=False)
display(pd.DataFrame({'check': ['Daily denominator dates', 'Daily/hourly overlapping dates'],
                      'matching_dates': [len(expected), len(hour_totals)], 'differences': [0, 0]}))

,check,matching_dates,differences
0,Daily denominator dates,638,0
1,Daily/hourly overlapping dates,92,0


## 5. Flag questionable periods without claiming a cause

An all-zero network date or a total below 20% of its past four same-weekday mean (at least two
available dates) triggers review. The threshold is a transparent screen, not an outage detector.
Holidays, weather and real disruptions can also lower demand. No weather causes are inferred.

Keep every published value. `analysis_boardings` is unavailable for review dates in the conservative
view; EDA compares that view with the inclusive totals. Ordinary route-level zeros remain zeros.

In [7]:
network = actual.rename('published_boardings').to_frame()
network['routes_reported'] = daily.groupby('payment_date').bus_route.nunique()
past = pd.concat([network.published_boardings.reindex(network.index - pd.Timedelta(days=k))
                  .set_axis(network.index).rename(str(k)) for k in [7, 14, 21, 28]], axis=1)
network['past_same_weekday_mean'] = past.mean(axis=1).where(past.notna().sum(axis=1).ge(2))
network['relative_to_past'] = network.published_boardings / network.past_same_weekday_mean.replace(0, np.nan)
network['all_zero_network'] = network.published_boardings.eq(0)
network['low_network_review'] = network.relative_to_past.lt(0.20)
network['demand_review_required'] = network.all_zero_network | network.low_network_review
daily = daily.merge(network[['all_zero_network', 'low_network_review', 'demand_review_required']],
                    on='payment_date', how='left', validate='many_to_one')
daily['analysis_boardings'] = daily.published_boardings.mask(daily.demand_review_required)
display(network.loc[network.demand_review_required])
print('Flagged dates:', int(network.demand_review_required.sum()))
print('Route/date rows retained for audit but masked for conservative analysis:', int(daily.demand_review_required.sum()))

,published_boardings,routes_reported,past_same_weekday_mean,relative_to_past,all_zero_network,low_network_review,demand_review_required
payment_date,,,,,,,
2026-01-10,0,427,698510.25,0.0,True,True,True
2026-01-11,0,427,556118.00,0.0,True,True,True
2026-01-12,0,427,1098420.00,0.0,True,True,True
2026-01-13,0,427,1137378.50,0.0,True,True,True
2026-01-14,0,427,1066480.50,0.0,True,True,True
2026-01-15,0,427,836538.00,0.0,True,True,True


Flagged dates: 6
Route/date rows retained for audit but masked for conservative analysis: 2562


In [8]:
hour_network = hourly.groupby('local_hour').ridership.sum().rename('ridership').to_frame()
prior_hours = pd.concat([hour_network.ridership.reindex(hour_network.index - pd.Timedelta(days=k))
                        .set_axis(hour_network.index).rename(str(k)) for k in [7, 14, 21, 28]], axis=1)
hour_network['past_same_weekday_hour_mean'] = prior_hours.mean(axis=1).where(prior_hours.notna().sum(axis=1).ge(2))
hour_network['relative_to_past'] = hour_network.ridership / hour_network.past_same_weekday_hour_mean.replace(0, np.nan)
hour_network['hour_review_required'] = hour_network.ridership.eq(0) | hour_network.relative_to_past.lt(0.20)
hourly = hourly.merge(hour_network[['hour_review_required']], on='local_hour', validate='many_to_one')
hourly['analysis_boardings'] = hourly.ridership.mask(hourly.hour_review_required)
hour_review_dates = hour_network.index[hour_network.hour_review_required].normalize().unique()
network['hourly_quality_screen_available'] = network.index.isin(hourly.payment_date.unique())
network['hourly_coverage_review'] = network.index.isin(hour_review_dates)
network['demand_review_required'] = network.demand_review_required | network.hourly_coverage_review
daily['hourly_quality_screen_available'] = daily.payment_date.map(network.hourly_quality_screen_available)
daily['hourly_coverage_review'] = daily.payment_date.map(network.hourly_coverage_review)
daily['demand_review_required'] = daily.payment_date.map(network.demand_review_required)
daily['analysis_boardings'] = daily.published_boardings.mask(daily.demand_review_required)
print('Hourly review flags:', int(hour_network.hour_review_required.sum()))
display(hour_network.loc[hour_network.hour_review_required])
print('True or unexplained route-level zeros are retained:', int(hourly.ridership.eq(0).sum()))
print('Daily conservative view also withholds dates containing a flagged observed hour:', list(hour_review_dates.strftime('%Y-%m-%d')))

Hourly review flags: 2


,ridership,past_same_weekday_hour_mean,relative_to_past,hour_review_required
local_hour,,,,
2026-08-16 21:00:00,195,20514.50,0.009505,True
2026-08-16 22:00:00,2391,14185.75,0.168549,True


True or unexplained route-level zeros are retained: 398896
Daily conservative view also withholds dates containing a flagged observed hour: ['2026-08-16']


## 6. Does each route have continuous observations?

Do not create ridership for absent route/dates. The coverage table distinguishes an observed
route's own date span from the full study window. The 92-day hourly pilot contains no daylight-saving
transition; a 24-hour grid check is appropriate only for this window.

In [9]:
coverage = daily.groupby('bus_route').agg(first_date=('payment_date', 'min'), last_date=('payment_date', 'max'),
    observed_dates=('payment_date', 'nunique'), zero_dates=('published_boardings', lambda s: s.eq(0).sum()),
    review_dates=('demand_review_required', 'sum'))
coverage['span_days'] = (coverage.last_date - coverage.first_date).dt.days + 1
coverage['absent_dates_inside_span'] = coverage.span_days - coverage.observed_dates
hour_counts = hourly.groupby(['bus_route', 'payment_date']).local_hour.nunique()
display(coverage.describe().round(2))
print('Daily rows absent inside an observed route span:', int(coverage.absent_dates_inside_span.sum()))
print('Hourly route-days with other than 24 rows:', int(hour_counts.ne(24).sum()))

,first_date,last_date,observed_dates,zero_dates,review_dates,span_days,absent_dates_inside_span
count,450,450,450.00,450.00,450.00,450.00,450.0
mean,2025-01-25 21:32:48,2026-09-30 00:00:00,613.10,158.06,6.69,613.10,0.0
min,2025-01-01 00:00:00,2026-09-30 00:00:00,126.00,6.00,1.00,126.00,0.0
25%,2025-01-01 00:00:00,2026-09-30 00:00:00,638.00,6.00,7.00,638.00,0.0
50%,2025-01-01 00:00:00,2026-09-30 00:00:00,638.00,6.00,7.00,638.00,0.0
75%,2025-01-01 00:00:00,2026-09-30 00:00:00,638.00,196.00,7.00,638.00,0.0
max,2026-05-28 00:00:00,2026-09-30 00:00:00,638.00,638.00,7.00,638.00,0.0
std,NaN,NaN,104.25,224.17,1.32,104.25,0.0


Daily rows absent inside an observed route span: 0
Hourly route-days with other than 24 rows: 0


## 7. Resolve Manhattan route identity from source records

Use **exact `routes.route_id` to `bus_route` equality**, checked in every dated feed and restricted
to bus routes referenced by trips. Both sources use `+` for the SBS codes in this collection.
The short names use `-SBS`, explaining the earlier short-name-only mismatch. This is a
versioned, dataset-specific crosswalk, not a global rule to replace strings for other agencies.

The primary schedule network is the Manhattan feed. M100 appears in the Bronx feed and remains
outside this historical pilot. Retain all other boroughs' demand for context; do not silently discard it.

In [10]:
mapping_rows = []
for path in sorted((RAW / 'gtfs/archive').glob('*.zip')):
    with zipfile.ZipFile(path) as z:
        r = pd.read_csv(z.open('routes.txt'), dtype='string')
        t = pd.read_csv(z.open('trips.txt'), dtype='string')
    referenced = r.loc[r.route_id.isin(t.route_id) & r.route_type.eq('3')].copy()
    assert referenced.route_id.is_unique and referenced.route_short_name.is_unique
    referenced['snapshot_id'] = path.stem
    referenced['bus_route'] = referenced.route_id
    referenced['exact_route_id_match'] = referenced.bus_route.isin(daily.bus_route)
    referenced['short_name_differs'] = referenced.route_short_name.ne(referenced.bus_route)
    mapping_rows.append(referenced[['snapshot_id', 'route_id', 'bus_route', 'route_short_name',
                                    'route_long_name', 'exact_route_id_match', 'short_name_differs']])
crosswalk = pd.concat(mapping_rows, ignore_index=True)
assert crosswalk.exact_route_id_match.all()
assert not crosswalk.duplicated(['snapshot_id', 'bus_route']).any()
pilot_routes = sorted(crosswalk.bus_route.unique())
display(crosswalk.groupby('snapshot_id').agg(routes=('bus_route', 'size'),
    exact_matches=('exact_route_id_match', 'sum'), SBS_name_differences=('short_name_differs', 'sum')))
display(crosswalk.loc[crosswalk.short_name_differs, ['route_id', 'route_short_name']].drop_duplicates())
print('Distinct historical Manhattan route codes:', len(pilot_routes))

,routes,exact_matches,SBS_name_differences
snapshot_id,,,
manhattan_20251230,41,41,9
manhattan_20260327,41,41,9
manhattan_20260409,41,41,9
manhattan_20260624,41,41,9
manhattan_20260903,41,41,9


,route_id,route_short_name
18,M14A+,M14A-SBS
19,M14D+,M14D-SBS
21,M15+,M15-SBS
25,M23+,M23-SBS
27,M34+,M34-SBS
28,M34A+,M34A-SBS
34,M60+,M60-SBS
37,M79+,M79-SBS
38,M86+,M86-SBS


Distinct historical Manhattan route codes: 41


## 8. Decisions and saved tables

Numeric whitespace in GTFS coordinates is parsed in working copies during feature construction.
No coordinates, times or route identifiers are guessed. Optional empty descriptive fields do not
become errors. No winsorization, synthetic passengers, global mean filling or crowding labels.

Cleaning is complete for the defined analysis view; unresolved real-world provenance and quality
questions remain explicitly flagged. Cleaning is not a claim that every source value is correct.

In [11]:
decisions = pd.DataFrame([
    ['Exact duplicate rows', 'Remove identical copies only; audit count', int(daily_duplicate_count + hourly_duplicate_count)],
    ['Conflicting keys', 'Quarantine all versions; stop for review', len(daily_conflicts) + len(hourly_conflicts)],
    ['Missing/invalid numeric fields', 'Stop; never replace with zero', 0],
    ['Low or zero network dates', 'Retain source values, mask conservative analysis', int(network.demand_review_required.sum())],
    ['Low or zero network hours', 'Retain source values, mask conservative analysis', int(hour_network.hour_review_required.sum())],
    ['SBS display names', 'Exact route_id crosswalk, version-specific', int(crosswalk.loc[crosswalk.short_name_differs, 'bus_route'].nunique())],
    ['Geometry numeric whitespace', 'Parse numeric working copies, preserve originals', 'audited in notebook 01'],
    ['Missing occupancy / actual arrivals', 'Unavailable; do not generate labels or delay metrics', 'not supplied']
], columns=['issue', 'decision', 'affected_count_or_status'])
display(decisions)
products = {'daily_clean': daily, 'hourly_clean': hourly, 'route_crosswalk': crosswalk,
            'route_date_coverage': coverage.reset_index(), 'network_date_quality': network.reset_index(),
            'network_hour_quality': hour_network.reset_index()}
for name, frame in products.items():
    frame.to_csv(OUT / f'{name}.csv', index=False)
decisions.to_csv(REPORT / 'cleaning_decisions.csv', index=False)
manifest = {'source_files': sources, 'products': {name: {'rows': len(frame), 'columns': list(frame.columns),
    'sha256': hashlib.sha256((OUT / f'{name}.csv').read_bytes()).hexdigest()} for name, frame in products.items()},
    'low_period_threshold': 0.20, 'past_comparison_days': [7,14,21,28],
    'interpretation': 'Review flags are heuristic; no data-outage cause is confirmed.'}
(OUT / 'cleaning_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
display(pd.DataFrame([{'product': name, 'rows': len(frame), 'columns': frame.shape[1]} for name, frame in products.items()]))
print('Saved six clean/audit tables, conflict tables and a fingerprinted manifest.')

,issue,decision,affected_count_or_status
0,Exact duplicate rows,Remove identical copies only; audit count,0
1,Conflicting keys,Quarantine all versions; stop for review,0
2,Missing/invalid numeric fields,Stop; never replace with zero,0
3,Low or zero network dates,"Retain source values, mask conservative analysis",7
4,Low or zero network hours,"Retain source values, mask conservative analysis",2
5,SBS display names,"Exact route_id crosswalk, version-specific",9
6,Geometry numeric whitespace,"Parse numeric working copies, preserve originals",audited in notebook 01
7,Missing occupancy / actual arrivals,Unavailable; do not generate labels or delay metrics,not supplied


,product,rows,columns
0,daily_clean,275896,12
1,hourly_clean,993600,8
2,route_crosswalk,205,7
3,route_date_coverage,450,8
4,network_date_quality,638,10
5,network_hour_quality,2208,5


Saved six clean/audit tables, conflict tables and a fingerprinted manifest.
